# Hand Gesture Recognition — ML Experiments (FINAL)

## Experiments
- **Experiment 1 — Hand Activity Identification** (target: `gesture_label`)
- **Experiment 2 — Person Identification** (target: `person_id`)

## Gesture label mapping
| Label | Gesture |
|-------|---------|
| 0 | up |
| 1 | down |
| 2 | play |
| 3 | pause |
| 4 | stop |

## Key Design Principle: Person-Isolated Splitting
All samples from the **same person always stay together** — either entirely in training or entirely in testing.
This prevents data leakage and ensures models generalise to **unseen persons**, not just unseen frames.

In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# GroupShuffleSplit: splits data keeping all rows of each group (person) together
# StratifiedGroupKFold: K-Fold that respects groups AND tries to balance classes
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold

from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score)
from sklearn.naive_bayes       import GaussianNB
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import SVC
import xgboost as xgb

print('All libraries imported successfully.')

All libraries imported successfully.


In [9]:
# ============================================================
# CELL 2 — LOAD DATA
# Replace path with your full dataset.
# ============================================================
df = pd.read_excel(r"C:\Users\User\Desktop\New Try\HAND GESTURE DETECTION Projest\New folder\New folder\hand_landmarks_updated\SMALL sample for test Hand gesture check.xlsx")  # ← change for full data

print(f'Dataset shape  : {df.shape}')
print(f'Columns        : {df.columns.tolist()}')
print(f'\nGesture labels : {sorted(df["gesture_label"].unique())}')
print(f'Unique persons : {df["person_id"].nunique()}')

# --- Duplicate check (important before any experiment) ---
dupes = df.duplicated().sum()
if dupes > 0:
    print(f'\n⚠️  WARNING: {dupes} duplicate rows found. Dropping them...')
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'   New shape after dropping: {df.shape}')
else:
    print(f'\n✅ No duplicate rows found.')

print(f'\nGesture distribution:\n{df["gesture_label"].value_counts().sort_index()}')
print(f'\nSamples per person:\n{df.groupby("person_id").size().to_string()}')

Dataset shape  : (322, 213)
Columns        : ['dist_0_1', 'dist_0_2', 'dist_0_3', 'dist_0_4', 'dist_0_5', 'dist_0_6', 'dist_0_7', 'dist_0_8', 'dist_0_9', 'dist_0_10', 'dist_0_11', 'dist_0_12', 'dist_0_13', 'dist_0_14', 'dist_0_15', 'dist_0_16', 'dist_0_17', 'dist_0_18', 'dist_0_19', 'dist_0_20', 'dist_1_2', 'dist_1_3', 'dist_1_4', 'dist_1_5', 'dist_1_6', 'dist_1_7', 'dist_1_8', 'dist_1_9', 'dist_1_10', 'dist_1_11', 'dist_1_12', 'dist_1_13', 'dist_1_14', 'dist_1_15', 'dist_1_16', 'dist_1_17', 'dist_1_18', 'dist_1_19', 'dist_1_20', 'dist_2_3', 'dist_2_4', 'dist_2_5', 'dist_2_6', 'dist_2_7', 'dist_2_8', 'dist_2_9', 'dist_2_10', 'dist_2_11', 'dist_2_12', 'dist_2_13', 'dist_2_14', 'dist_2_15', 'dist_2_16', 'dist_2_17', 'dist_2_18', 'dist_2_19', 'dist_2_20', 'dist_3_4', 'dist_3_5', 'dist_3_6', 'dist_3_7', 'dist_3_8', 'dist_3_9', 'dist_3_10', 'dist_3_11', 'dist_3_12', 'dist_3_13', 'dist_3_14', 'dist_3_15', 'dist_3_16', 'dist_3_17', 'dist_3_18', 'dist_3_19', 'dist_3_20', 'dist_4_5', 'dist_4_6'

In [10]:
# ============================================================
# CELL 3 — GLOBAL CONFIGURATION
# ============================================================

# Train-test split ratios
SPLIT_RATIOS = {
    '80-20': 0.20,
    '70-30': 0.30,
    '60-40': 0.40,
    '50-50': 0.50
}

# Percentage of total data to use
DATA_PERCENTAGES = [100, 75, 50, 25]

# Max folds for K-Fold CV
# We use min(5, n_unique_persons) to handle small datasets or small subsets
MAX_FOLDS = 5

print('Configuration set.')

Configuration set.


In [11]:
# ============================================================
# CELL 4 — UTILITY FUNCTIONS
# ============================================================

def get_models():
    """
    Returns a fresh dict of 7 classifiers each time it is called.
    Calling inside a loop ensures each iteration starts with an untrained model.

    XGBoost special requirements:
      - eval_metric='mlogloss'  → silences default metric warning
      - verbosity=0             → silences all XGBoost output
      - Labels MUST be 0-indexed integers → always use remap_labels_for_xgboost()
    """
    return {
        'Naive Bayes'        : GaussianNB(),
        'Decision Tree'      : DecisionTreeClassifier(random_state=42),
        'XGBoost'            : xgb.XGBClassifier(
                                    eval_metric='mlogloss',
                                    verbosity=0,
                                    random_state=42),
        'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
        'KNN'                : KNeighborsClassifier(n_neighbors=5),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'SVM'                : SVC(kernel='rbf', random_state=42)
    }


def remap_labels_for_xgboost(y):
    """
    XGBoost requires class labels to be exactly 0, 1, 2, ...
    This is critical for Experiment 2 where person_id values are
    arbitrary integers (e.g. 27, 50, 103...).

    IMPORTANT: Build the mapping from y_train, then apply the SAME
    mapping to y_test. Never build separate mappings for train and test.

    Returns:
        y_remapped : array of 0-indexed integers
        label_map  : {original → new} dict to apply to test labels
    """
    unique_labels = sorted(np.unique(y))
    label_map = {orig: new for new, orig in enumerate(unique_labels)}
    return np.array([label_map[val] for val in y]), label_map


def get_metrics(y_true, y_pred):
    """
    Returns (accuracy, precision, recall, f1) all as percentages.
    'weighted' average handles multi-class correctly.
    zero_division=0 prevents errors when a class has no predictions.
    """
    acc  = round(accuracy_score (y_true, y_pred)                                       * 100, 2)
    prec = round(precision_score(y_true, y_pred, average='weighted', zero_division=0)  * 100, 2)
    rec  = round(recall_score   (y_true, y_pred, average='weighted', zero_division=0)  * 100, 2)
    f1   = round(f1_score       (y_true, y_pred, average='weighted', zero_division=0)  * 100, 2)
    return acc, prec, rec, f1


def get_data_subset(df, pct, random_state=42):
    """
    Returns pct% of the dataframe by sampling 'pct%' of rows from EACH
    person separately. This maintains the person structure.

    Why sample per person (not globally)?
    - Global random sampling might under-sample some persons
    - Sampling per person ensures every person contributes proportionally
    - The person_id column is preserved for group-based splitting

    NOTE: We do NOT shuffle here — GroupShuffleSplit handles randomisation.
    """
    if pct == 100:
        return df.copy().reset_index(drop=True)

    frac = pct / 100.0
    frames = []
    for pid, grp in df.groupby('person_id'):
        frames.append(grp.sample(frac=frac, random_state=random_state))
    return pd.concat(frames).reset_index(drop=True)


def person_isolated_split(X, y, groups, test_size, random_state=42):
    """
    Splits data so that ALL rows belonging to a person go ENTIRELY
    to train or ENTIRELY to test — never both.

    Why GroupShuffleSplit instead of StratifiedShuffleSplit?
    -------------------------------------------------------
    StratifiedShuffleSplit splits ROWS randomly while keeping
    class proportions. It DOES NOT know about persons, so the
    same person's gestures can appear in both train and test.

    This causes DATA LEAKAGE:
    - Model sees person A's gestures in training
    - Tests on different gestures from the SAME person A
    - Model memorises person-specific hand shape → inflated accuracy
    - Does NOT generalise to NEW (unseen) persons

    GroupShuffleSplit respects groups (persons) → zero leakage.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


print('All utility functions defined.')

All utility functions defined.


---
# EXPERIMENT 1 — Hand Activity Identification

**Target:** `gesture_label` | **Drop:** `person_name`, `person_id`

**Splitting strategy:** `GroupShuffleSplit` with `groups = person_id`
- Every person's data stays entirely in train OR test
- Ensures the model learns gesture patterns that work for **new persons**
- This is the scientifically correct way to evaluate a gesture recogniser

**Cross-Validation:** `StratifiedGroupKFold`
- Respects person groups (no leakage across folds)
- Tries to maintain class (gesture) balance per fold

In [12]:
# ============================================================
# CELL 5 — EXPERIMENT 1: Hand Activity Identification
# ============================================================

# Drop person columns — keep only hand distance features and target
FEATURE_COLS_EXP1 = [c for c in df.columns
                     if c not in ['person_name', 'person_id', 'gesture_label']]
TARGET_EXP1 = 'gesture_label'

results_exp1    = []   # train/test split results
cv_results_exp1 = []   # K-Fold CV results

for data_pct in DATA_PERCENTAGES:
    print(f"\n{'='*65}")
    print(f"  [EXP 1]  {data_pct}% of Total Data")
    print(f"{'='*65}")

    # Get subset — pct% of rows from each person (maintains person structure)
    df_sub   = get_data_subset(df, data_pct)
    X_sub    = df_sub[FEATURE_COLS_EXP1].values
    y_sub    = df_sub[TARGET_EXP1].values
    # groups = person_id for each row — used by GroupShuffleSplit & StratifiedGroupKFold
    groups   = df_sub['person_id'].values
    n_persons = len(np.unique(groups))

    print(f"  Rows          : {len(df_sub)}")
    print(f"  Unique persons: {n_persons}")
    print(f"  Gesture dist  : {dict(zip(*np.unique(y_sub, return_counts=True)))}")

    # --------------------------------------------------------
    # PART A: Train / Test Splits (4 ratios × 7 models)
    # --------------------------------------------------------
    for ratio_label, test_size in SPLIT_RATIOS.items():
        print(f"\n  --- Split {ratio_label} ---")

        # Person-isolated split: all rows of a person go to train OR test only
        X_train, X_test, y_train, y_test = person_isolated_split(
            X_sub, y_sub, groups, test_size=test_size
        )
        print(f"  Train={len(y_train)}, Test={len(y_test)}")

        for model_name, model in get_models().items():
            try:
                if model_name == 'XGBoost':
                    # gesture_label is 0-4 so remapping is technically optional here,
                    # but we always remap for consistency and safety
                    y_tr_xgb, lmap = remap_labels_for_xgboost(y_train)
                    y_te_xgb       = np.array([lmap[v] for v in y_test])
                    model.fit(X_train, y_tr_xgb)
                    y_pred = model.predict(X_test)
                    acc, prec, rec, f1 = get_metrics(y_te_xgb, y_pred)
                else:
                    model.fit(X_train, y_train)
                    y_pred = model.predict(X_test)
                    acc, prec, rec, f1 = get_metrics(y_test, y_pred)

                print(f"  {model_name:<22} Acc={acc}%  Prec={prec}%  Rec={rec}%  F1={f1}%")
                results_exp1.append({
                    'Total Data %'    : data_pct,
                    'Train-Test Ratio': ratio_label,
                    'Algorithm'       : model_name,
                    'Accuracy'        : acc,
                    'Precision'       : prec,
                    'Recall'          : rec,
                    'F1-Score'        : f1
                })
            except Exception as e:
                print(f"  {model_name:<22} ERROR: {e}")

    # --------------------------------------------------------
    # PART B: StratifiedGroupKFold Cross-Validation
    #
    # Why StratifiedGroupKFold instead of StratifiedKFold?
    #   - StratifiedKFold splits rows while maintaining class balance,
    #     but it does NOT respect person groups → same person in train & test
    #   - StratifiedGroupKFold maintains both:
    #       1. Group integrity (person never split across folds)
    #       2. Class balance as much as possible
    #
    # n_folds = min(MAX_FOLDS, n_persons):
    #   - Cannot have more folds than unique persons
    #     (each fold needs at least 1 person in test)
    # --------------------------------------------------------
    n_folds = min(MAX_FOLDS, n_persons)
    print(f"\n  --- StratifiedGroupKFold ({n_folds}-Fold) CV on {data_pct}% data ---")

    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=42)

    for model_name, model in get_models().items():
        fold_acc, fold_prec, fold_rec, fold_f1 = [], [], [], []

        for fold_num, (train_idx, test_idx) in enumerate(
                sgkf.split(X_sub, y_sub, groups=groups), 1):

            X_tr, X_te = X_sub[train_idx], X_sub[test_idx]
            y_tr, y_te = y_sub[train_idx],  y_sub[test_idx]

            try:
                if model_name == 'XGBoost':
                    # Remap using ONLY this fold's training labels
                    # (different folds may see different label subsets)
                    y_tr_xgb, lmap = remap_labels_for_xgboost(y_tr)
                    y_te_xgb       = np.array([lmap[v] for v in y_te])
                    model.fit(X_tr, y_tr_xgb)
                    y_pred = model.predict(X_te)
                    acc, prec, rec, f1 = get_metrics(y_te_xgb, y_pred)
                else:
                    model.fit(X_tr, y_tr)
                    y_pred = model.predict(X_te)
                    acc, prec, rec, f1 = get_metrics(y_te, y_pred)

                fold_acc.append(acc);  fold_prec.append(prec)
                fold_rec.append(rec);  fold_f1.append(f1)

            except Exception as e:
                print(f"  {model_name:<22} Fold {fold_num} ERROR: {e}")
                fold_acc.append(np.nan); fold_prec.append(np.nan)
                fold_rec.append(np.nan); fold_f1.append(np.nan)

        # Aggregate mean ± std across all folds
        mean_acc  = round(np.nanmean(fold_acc),  2)
        mean_prec = round(np.nanmean(fold_prec), 2)
        mean_rec  = round(np.nanmean(fold_rec),  2)
        mean_f1   = round(np.nanmean(fold_f1),   2)
        std_acc   = round(np.nanstd(fold_acc),   2)

        print(f"  {model_name:<22} CV Acc={mean_acc}%±{std_acc}%  "
              f"Prec={mean_prec}%  Rec={mean_rec}%  F1={mean_f1}%")

        cv_results_exp1.append({
            'Total Data %' : data_pct,
            'N Folds'      : n_folds,
            'Algorithm'    : model_name,
            'CV Acc Mean'  : mean_acc,
            'CV Acc Std'   : std_acc,
            'CV Prec Mean' : mean_prec,
            'CV Prec Std'  : round(np.nanstd(fold_prec), 2),
            'CV Rec Mean'  : mean_rec,
            'CV Rec Std'   : round(np.nanstd(fold_rec),  2),
            'CV F1 Mean'   : mean_f1,
            'CV F1 Std'    : round(np.nanstd(fold_f1),   2)
        })

print("\n✅ Experiment 1 Complete!")


  [EXP 1]  100% of Total Data
  Rows          : 322
  Unique persons: 6
  Gesture dist  : {np.int64(0): np.int64(69), np.int64(1): np.int64(60), np.int64(2): np.int64(60), np.int64(3): np.int64(73), np.int64(4): np.int64(60)}

  --- Split 80-20 ---
  Train=200, Test=122
  Naive Bayes            Acc=99.18%  Prec=99.21%  Rec=99.18%  F1=99.18%
  Decision Tree          Acc=70.49%  Prec=79.63%  Rec=70.49%  F1=67.79%
  XGBoost                Acc=77.05%  Prec=81.27%  Rec=77.05%  F1=76.36%
  Random Forest          Acc=85.25%  Prec=89.4%  Rec=85.25%  F1=84.64%
  KNN                    Acc=99.18%  Prec=99.21%  Rec=99.18%  F1=99.18%
  Logistic Regression    Acc=97.54%  Prec=97.77%  Rec=97.54%  F1=97.54%
  SVM                    Acc=95.08%  Prec=95.93%  Rec=95.08%  F1=95.07%

  --- Split 70-30 ---
  Train=200, Test=122
  Naive Bayes            Acc=99.18%  Prec=99.21%  Rec=99.18%  F1=99.18%
  Decision Tree          Acc=70.49%  Prec=79.63%  Rec=70.49%  F1=67.79%
  XGBoost                Acc=77.05% 

In [13]:
# ============================================================
# CELL 6 — EXP 1: Display Results
# ============================================================

ALGO_ORDER   = ['Naive Bayes', 'Decision Tree', 'XGBoost', 'Random Forest',
                'KNN', 'Logistic Regression', 'SVM']
METRIC_ORDER = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

df_exp1 = pd.DataFrame(results_exp1)

pivot_exp1 = df_exp1.pivot_table(
    index   =['Total Data %', 'Train-Test Ratio'],
    columns ='Algorithm',
    values  =METRIC_ORDER
)
pivot_exp1 = pivot_exp1.reindex(
    columns=pd.MultiIndex.from_product([METRIC_ORDER, ALGO_ORDER])
)

print('Experiment 1 — Train/Test Results (person-isolated split):')
print(pivot_exp1.to_string())

df_cv1 = pd.DataFrame(cv_results_exp1)
print('\nExperiment 1 — StratifiedGroupKFold CV Results:')
print(df_cv1.to_string(index=False))

Experiment 1 — Train/Test Results (person-isolated split):
                                 Accuracy                                                                          Precision                                                                             Recall                                                                           F1-Score                                                                       
                              Naive Bayes Decision Tree XGBoost Random Forest    KNN Logistic Regression     SVM Naive Bayes Decision Tree XGBoost Random Forest    KNN Logistic Regression     SVM Naive Bayes Decision Tree XGBoost Random Forest    KNN Logistic Regression     SVM Naive Bayes Decision Tree XGBoost Random Forest    KNN Logistic Regression     SVM
Total Data % Train-Test Ratio                                                                                                                                                                                            